# 04 — Test Pipeline Modules

**No output artifact.** Purpose: validate every `src/` module works correctly
in isolation and in sequence, before wiring into `app.py`.

Run this notebook after implementing each new module.

> Requires: `data/chroma_db/` (from notebook 03) for sections 5 and 6.

---

## Sections
1. TextCleaner
2. RuleFilter
3. ConversationMemoryManager
4. DecisionEngine (with MockQueryClassifier)
5. Retriever
6. RAGChain
7. PostGenerationChecker
8. ControlledResponseManager
9. Full pipeline walkthrough — one question through every stage

In [ ]:
import sys
sys.path.append('..')

print('Imports OK')

---
## 1 — TextCleaner

In [ ]:
# from src.preprocessing.text_cleaner import TextCleaner
# cleaner = TextCleaner()

# assert cleaner.clean('  bonjour  ') == 'bonjour'
# assert cleaner.clean('mon   père  est   intubé') == 'mon père est intubé'
# assert 'réanimation' in cleaner.clean('service de réanimation')
# try:
#     cleaner.clean('')
#     assert False, 'Should have raised ValueError'
# except ValueError:
#     pass
# print('TextCleaner: OK')

---
## 2 — RuleFilter

In [ ]:
# from src.safety.rule_filter import RuleFilter
# rf = RuleFilter()

# cases = [
#     ('Est-ce que mon père va survivre ?',  'prognosis',  True),
#     ('Quel médicament donner ?',            'treatment',  True),
#     ('Il ne respire plus',                  'urgency',    True),
#     ("Qu'est-ce qu'un respirateur ?",       'allowed',    False),
# ]
# for text, expected_cat, expected_blocked in cases:
#     cat, blocked = rf.filter(text)
#     status = 'OK' if cat == expected_cat and blocked == expected_blocked else 'FAIL'
#     print(f'[{status}] "{text[:40]}" → ({cat}, blocked={blocked})')

---
## 3 — ConversationMemoryManager

In [ ]:
# from src.memory.conversation_memory import ConversationMemoryManager
# mem = ConversationMemoryManager()

# mem.add_user_message("Qu'est-ce que la sédation ?")
# mem.add_assistant_message('La sédation est une technique qui...')
# mem.add_user_message('Est-ce grave ?')

# history = mem.get_history()
# assert len(history) == 3
# print(f'History ({len(history)} turns):')
# for turn in history:
#     print(f'  [{turn["role"]}] {turn["content"][:60]}')

# context = mem.get_context_string()
# print(f'\nContext string for classifier:\n{context}')

# mem.clear()
# assert mem.get_history() == []
# print('\nAfter clear: OK')

---
## 4 — DecisionEngine with MockQueryClassifier

Tests all 4 decision outcomes without a trained model.

In [ ]:
# from src.safety.rule_filter import RuleFilter
# from src.safety.query_classifier import MockQueryClassifier
# from src.safety.decision_engine import DecisionEngine

# engine = DecisionEngine(rule_filter=RuleFilter(), classifier=MockQueryClassifier())

# test_cases = [
#     ('Il ne respire plus',              'redirect'),
#     ('Quel médicament donner ?',        'refuse'),
#     ("Qu'est-ce qu'un respirateur ?",  'allow_rag'),
# ]
# for text, expected in test_cases:
#     decision = engine.decide(text)
#     status = 'OK' if decision == expected else 'FAIL'
#     print(f'[{status}] "{text[:40]}" → {decision} (expected {expected})')

---
## 5 — Retriever

> Requires `data/chroma_db/` — run notebook 03 first.

In [ ]:
# from src.rag.vector_store import VectorStoreManager
# from src.rag.retriever import Retriever

# store = VectorStoreManager().get_store()
# retriever = Retriever(store)

# results = retriever.search("Qu'est-ce que la sédation ?")
# print(f'Results for known query: {len(results)}')
# for r in results:
#     print(f'  [{r["score"]:.3f}] {r["source"]} — {r["text"][:80]}...')

# no_results = retriever.search('meilleur restaurant de Paris')
# assert no_results == [], f'Expected empty, got {no_results}'
# print('Unknown query: correctly empty')

---
## 6 — RAGChain

> Requires `OPENAI_API_KEY` in `.env` and `data/chroma_db/`.

In [ ]:
# from src.rag.rag_chain import RAGChain

# rag = RAGChain(retriever)
# answer, sources = rag.generate("Qu'est-ce que la sédation ?")

# print(f'Answer:\n{answer}')
# print(f'\nSources: {sources}')
# assert sources, 'Sources must not be empty for an answered question'
# assert 'sédation' in answer.lower() or 'endormi' in answer.lower(), 'Answer should mention sedation'

---
## 7 — PostGenerationChecker

In [ ]:
# from src.safety.post_checker import PostGenerationChecker
# checker = PostGenerationChecker()

# safe_answer = 'La sédation est une technique générale utilisée en réanimation pour...(source: doc1.pdf)'
# result, modified = checker.check(safe_answer)
# assert not modified, 'Safe answer should not be modified'
# print(f'Safe answer: not modified — OK')

# dangerous_answers = [
#     'Il va survivre, ne vous inquiétez pas.',
#     'Donnez-lui 500mg de paracétamol.',
#     'Selon moi, son état va empirer.',
# ]
# for ans in dangerous_answers:
#     _, modified = checker.check(ans)
#     status = 'OK' if modified else 'FAIL'
#     print(f'[{status}] Dangerous answer correctly rejected: "{ans[:50]}"')

---
## 8 — ControlledResponseManager

In [ ]:
# from src.safety.controlled_responses import ControlledResponseManager
# crm = ControlledResponseManager()

# responses = {
#     'urgency':        crm.get_urgency_redirect(),
#     'medical_advice': crm.get_refuse_medical_advice(),
#     'reformulation':  crm.get_ask_reformulation(),
#     'no_source':      crm.get_no_source_found(),
#     'out_of_scope':   crm.get_out_of_scope(),
# }
# for name, resp in responses.items():
#     assert resp, f'{name} returned empty string'
#     print(f'[{name}]\n{resp}\n')

---
## 9 — Full pipeline walkthrough

Trace one question through every stage in sequence and print output at each step.

In [ ]:
QUESTION = "Qu'est-ce que la sédation en réanimation ?"
print(f'Question: {QUESTION}\n')

# --- Stage 1: TextCleaner
# cleaned = cleaner.clean(QUESTION)
# print(f'1. Cleaned:   {cleaned}')

# --- Stage 2: Add to memory (simulating 2nd turn)
# mem.add_user_message(cleaned)
# context = mem.get_context_string()
# print(f'2. Context:   {context[:80]}...')

# --- Stage 3: RuleFilter
# cat, blocked = rf.filter(cleaned)
# print(f'3. RuleFilter: category={cat}, blocked={blocked}')

# --- Stage 4: DecisionEngine
# decision = engine.decide(cleaned)
# print(f'4. Decision:  {decision}')

# --- Stage 5: Route
# if decision == 'allow_rag':
#     results = retriever.search(cleaned)
#     print(f'5. Retrieved: {len(results)} chunks')
#     answer, sources = rag.generate(cleaned)
#     answer, modified = checker.check(answer)
#     print(f'6. Answer (post-check, modified={modified}):\n   {answer[:200]}')
#     print(f'7. Sources:   {sources}')
# else:
#     response = crm.get_refuse_medical_advice()
#     print(f'5. Controlled response:\n   {response}')